# Assignment 2: The Dirty Ledger
## Data Inspection, AI-Assisted Cleaning and Verification

This notebook inspects `teaching_prices_messy.csv`, tests possible data-quality issues, creates a separate cleaned copy, and verifies that the original source file remains unchanged.

**Important:** Run the cells from top to bottom. Keep `teaching_prices_messy.csv` in the same folder as this notebook, or update the file path in the loading cell. Do not invent or manually type outputs; use the results produced by your own dataset.

In [ ]:
import pandas as pd
import hashlib

In [ ]:
file_path = "teaching_prices_messy.csv"
raw = pd.read_csv(file_path)

print("Dataset loaded successfully.")
print("Number of rows and columns:", raw.shape)

display(raw.head(10))

In [ ]:
# Calculate the SHA-256 hash of the original CSV file.
with open(file_path, "rb") as file:
    original_hash = hashlib.sha256(file.read()).hexdigest()

print("Original file SHA-256:")
print(original_hash)

## 1. Initial inspection

The following checks inspect the dataset before making any changes.

In [ ]:
raw.info()

In [ ]:
print("Missing values in each column:")
display(raw.isna().sum())

In [ ]:
print("Number of exact duplicate rows:", raw.duplicated().sum())

print("Exact duplicate rows:")
display(raw.loc[raw.duplicated(keep=False)].sort_values(list(raw.columns)))

In [ ]:
print("Price values and their frequencies:")
display(raw["price"].value_counts(dropna=False))

In [ ]:
# Test direct numeric conversion to identify values that cannot be read as numbers.
price_test = pd.to_numeric(raw["price"], errors="coerce")

invalid_price_values = raw.loc[
    raw["price"].notna() & price_test.isna(), ["price"]
].drop_duplicates()

print("Non-missing price values that fail direct numeric conversion:")
display(invalid_price_values)

In [ ]:
# Test a comma-removal step separately before changing the working copy.
price_check = raw["price"].astype("string").str.replace(",", "", regex=False)
price_check = pd.to_numeric(price_check, errors="coerce")

print("Highest numeric prices after removing commas:")
display(price_check.sort_values(ascending=False).head(10))

In [ ]:
print("Item label frequencies:")
display(raw["item"].value_counts(dropna=False))

print("Distinct item labels:")
display(raw["item"].drop_duplicates().sort_values())

## 2. Cleaning decisions to be checked

- Remove exact duplicate rows from the working copy.
- Remove commas from price strings before converting prices to numeric values.
- Keep missing prices missing; do not guess or fill them.
- Flag unusually high prices for review; do not automatically delete them.
- Inspect inconsistent item labels; do not silently rename them.
- Convert the date column to a datetime type and check for invalid dates.
- Keep the original `raw` DataFrame unchanged.

In [ ]:
# Create a separate working copy. The original raw DataFrame is preserved.
clean = raw.copy(deep=True)

print("Working copy created.")
print("Original shape:", raw.shape)
print("Working-copy shape:", clean.shape)

In [ ]:
print("Exact duplicate rows before removal:")
display(clean.loc[clean.duplicated(keep=False)].sort_values(list(clean.columns)))

clean = clean.drop_duplicates()

print("Shape after removing exact duplicates:", clean.shape)
print("Exact duplicate rows remaining:", clean.duplicated().sum())

In [ ]:
# Remove thousands separators, then convert prices to numeric.
clean["price"] = clean["price"].astype("string").str.replace(",", "", regex=False)
clean["price"] = pd.to_numeric(clean["price"], errors="coerce")

print("Price column converted to numeric.")
print("Missing prices after conversion:", clean["price"].isna().sum())
display(clean[["price"]].head(10))

In [ ]:
# Convert date strings to datetime values.
clean["date"] = pd.to_datetime(clean["date"], errors="coerce")

print("Date column converted.")
print("Invalid or missing dates:", clean["date"].isna().sum())

In [ ]:
# Create a flag so missing prices remain visible for review.
clean["missing_price_flag"] = clean["price"].isna()

print("Rows with missing prices:")
display(clean.loc[clean["missing_price_flag"]])

In [ ]:
# Flag high prices for investigation; do not remove them automatically.
clean["high_price_flag"] = clean["price"] >= 1000

print("Rows flagged for high prices:")
display(clean.loc[clean["high_price_flag"]])

In [ ]:
# Flag item labels that differ from the exact spelling "Oil B"
# after ignoring surrounding spaces and letter case.
# The original item text is preserved.
clean["label_flag"] = (
    clean["item"].notna()
    & (clean["item"].astype("string").str.strip().str.lower() == "oil b")
    & (clean["item"] != "Oil B")
)

print("Rows with a possible Oil B label inconsistency:")
display(clean.loc[clean["label_flag"], ["item", "city", "date", "price"]])

In [ ]:
print("Cleaned working-copy information:")
clean.info()

In [ ]:
print("Missing values after cleaning:")
display(clean.isna().sum())

print("Remaining exact duplicate rows:", clean.duplicated().sum())

print("Rows with invalid dates:")
display(clean.loc[clean["date"].isna()])

In [ ]:
print("Original shape:", raw.shape)
print("Working-copy shape after cleaning:", clean.shape)

print("Exact duplicates in original:", raw.duplicated().sum())
print("Exact duplicates in working copy:", clean.duplicated().sum())

print("Missing prices in original:", raw["price"].isna().sum())
print("Missing prices in working copy:", clean["price"].isna().sum())

In [ ]:
print("Rows requiring review: missing prices")
display(clean.loc[clean["missing_price_flag"]])

print("Rows requiring review: high prices")
display(clean.loc[clean["high_price_flag"]])

print("Rows requiring review: possible label inconsistency")
display(clean.loc[clean["label_flag"]])

## 3. Save the cleaned working copy

The cleaned file is exported separately. The original CSV is not overwritten.

In [ ]:
output_path = "teaching_prices_cleaned.csv"
clean.to_csv(output_path, index=False)

print("Cleaned dataset saved as:", output_path)

In [ ]:
# Verify that the original source file has not changed.
with open(file_path, "rb") as file:
    final_hash = hashlib.sha256(file.read()).hexdigest()

print("Original SHA-256:", original_hash)
print("Final SHA-256:   ", final_hash)
print("Original file unchanged:", original_hash == final_hash)